# FROST Hydrogel Model — complete Google Colab notebook

This notebook consolidates the Hydrogel model code in one place: nominal simulation, drying/pH/growth/attachment dynamics, post-removal persistence, storage scenarios, carrying-capacity and CFU-conversion sweeps, seven-day feasibility envelope, Morris screening, 15-input Sobol/Jansen analysis, fresh-vs-storage uncertainty propagation, and the relative cargo interface.

**Important:** the cargo section requires the actual continuous `P_trxB(t)` trajectory exported from the upstream sensing model. The notebook does not invent that input.

## Cross-model ownership and synchronization

- **FLARE owns** the upstream promoter trajectory `P_trxB(t)`.
- **Hydrogel owns** bacterial viability/attachment plus the normalized intracellular cargo state `r(t)` and loading state `theta(t)`. The useful cargo equation is `dr/dt = k_deg [P_trxB(t) f_aw,sec - r]`; the normalized formulation does not require a separate cargo-synthesis scale or reference-pool parameter.
- **QUENCH owns** extracellular uptake, endosomal escape, RISC loading and STAT6/RelA silencing. The current QUENCH source `alpha_sec * P_trxB(t)` is a standalone normalized effective-source calibration, not a measured absolute Hydrogel output. A future quantitative handoff requires a measured scale factor from Hydrogel relative loaded-cargo output to QUENCH effective concentration.

The sensitivity section below uses the frozen **15-input** prior set used in the final Hydrogel analysis.


## 0. Imports, constants and canonical parameters

In [ ]:
import math
from collections import OrderedDict
import numpy as np
from scipy.integrate import solve_ivp
from scipy.stats import qmc

V0=1.0; A_SKIN=25.0; RHO_W=1.0; MW_H2O=18.015; TEWL=40.0; RH=0.50; T_SKIN=32.0
J_SKIN=TEWL*(A_SKIN*1e-4)
MU_OPT=0.14; T_MIN,T_OPT,T_MAX=8.0,37.0,45.0; PH_MIN,PH_OPT,PH_MAX=3.4,6.0,8.0
K_S=0.5; Y_XS=0.20; M_S_MAINT=0.0025; Y_LA_S=0.0123; K_HL=10e-3
A_W_SKIN=0.70; K_D0=8e-4; K_ACID=0.50
KA_LAC=10**(-3.86); KA_GLUC=10**(-3.86); MW_FOS=666.0
N_GLYCEROL=0.016/92.094; N_NA_LAC=0.134e-3; N_CA=0.045e-3; N_GLUC=0.090e-3; N_K=0.0067e-3; N_GLYC=0.0033e-3; N_NA_ALG=0.076e-3
M0_PRE_STORAGE=0.001; MS0=0.020; NL0=0.13913e-3
K_DEG=math.log(2.0)/24.0
BASE=dict(V_nonwater=0.20,k_E=0.118,k_desicc=0.015,k_att=0.020,A_max=1e-4,k_shed=0.004,k_strip=0.010,k_starv=0.005,k_host=0.005,T_gel=48.0,a_w_min=0.90,a_w_lethal=0.85,a_w_sec=0.65,K_cap=1.2,c_CFU=1e12,k_store=0.0,floor=1e4)

def clip01(x): return max(0.0,min(1.0,x))
def f_temperature(T):
    numerator=(T-T_MAX)*(T-T_MIN)**2
    denominator=(T_OPT-T_MIN)*((T_OPT-T_MIN)*(T-T_OPT)-(T_OPT-T_MAX)*(T_OPT+T_MIN-2*T))
    return numerator/denominator
F_T=f_temperature(T_SKIN)

def pH_and_HL(V_mL,N_lactate):
    V_L=max(V_mL,1e-12)/1000.0
    strong_excess=(N_NA_LAC+2*N_CA+N_K-2*N_GLYC)/V_L
    total_weak=(max(N_lactate,0.0)+N_GLUC)/V_L
    H=KA_LAC*(total_weak/strong_excess-1.0); H=max(H,1e-12)
    pH=-math.log10(H); L_T=max(N_lactate,0.0)/V_L; HL=L_T*H/(KA_LAC+H)
    return pH,HL

def aw_value(V,M_S,N_L,V_nonwater):
    n_water=max((V-V_nonwater)*RHO_W/MW_H2O,1e-15)
    n_osm=N_GLYCEROL+max(M_S,0.0)/MW_FOS+max(N_L,0.0)+N_NA_LAC+N_CA+N_GLUC+N_K+N_GLYC+N_NA_ALG
    return n_water/(n_water+n_osm)

def model_rates(state,p):
    V,M_G,M_A,M_S,N_L=state
    V=max(V,p['V_nonwater']+1e-10); M_G=max(M_G,0.0); M_A=max(M_A,0.0); M_S=max(M_S,0.0); N_L=max(N_L,0.0)
    aw=aw_value(V,M_S,N_L,p['V_nonwater']); pH,HL=pH_and_HL(V,N_L)
    B=1000*M_G/V; S=1000*M_S/V
    f_aw=math.sqrt(clip01((aw-p['a_w_min'])/(1-p['a_w_min'])))
    nph=(pH-PH_MIN)*(pH-PH_MAX); dph=nph-(pH-PH_OPT)**2; f_pH=clip01(nph/dph if dph!=0 else 0)
    f_HL=1/(1+HL/K_HL)
    mu=MU_OPT*F_T*S/(K_S+S)*max(0,1-B/p['K_cap'])*f_aw*f_pH*f_HL
    den=max(p['a_w_min']-p['a_w_lethal'],1e-9)
    g=clip01((p['a_w_min']-aw)/den)**2
    kd=K_D0+p['k_desicc']*g+K_ACID*HL
    attach=p['k_att']*M_G*max(0,1-M_A/p['A_max'])
    dV=J_SKIN*(1-aw)-p['k_E']*max(0,aw-RH)
    dMG=(mu-kd)*M_G-attach
    dMA=attach-kd*M_A
    dMS=-(mu/Y_XS+M_S_MAINT)*M_G
    dNL=Y_LA_S*(mu*(1/Y_XS-1)+M_S_MAINT)*M_G
    f_aw_sec=math.sqrt(clip01((aw-p['a_w_sec'])/(p['a_w_min']-p['a_w_sec'])))
    return np.array([dV,dMG,dMA,dMS,dNL]),dict(aw=aw,pH=pH,HL=HL,B=B,S=S,f_aw=f_aw,f_pH=f_pH,f_HL=f_HL,mu=mu,g_desicc=g,k_d_G=kd,attach=attach,f_aw_sec=f_aw_sec)

def simulate_hydrogel(params=None,shelf_days=0.0,points_per_hour=100,initial_biomass_g=None):
    p=BASE.copy(); p.update(params or {})
    if initial_biomass_g is None:
        MG0=M0_PRE_STORAGE*math.exp(-p['k_store']*shelf_days)
    else: MG0=initial_biomass_g
    y0=[V0,MG0,0,MS0,NL0]
    def rhs(t,y): return model_rates(y,p)[0]
    t_eval=np.linspace(0,p['T_gel'],max(2,int(p['T_gel']*points_per_hour)+1))
    sol=solve_ivp(rhs,[0,p['T_gel']],y0,method='LSODA',t_eval=t_eval,rtol=2e-6,atol=[1e-9,1e-12,1e-12,1e-10,1e-12],max_step=.2)
    if not sol.success: raise RuntimeError(sol.message)
    deriv=[model_rates(sol.y[:,i],p)[1] for i in range(sol.t.size)]
    out={'t_h':sol.t,'V':sol.y[0],'M_G':sol.y[1],'M_A':sol.y[2],'M_S':sol.y[3],'N_L':sol.y[4],'params':p}
    for k in deriv[0]: out[k]=np.array([d[k] for d in deriv])
    out['N_G']=out['M_G']*p['c_CFU']; out['sigma_A']=out['M_A']*p['c_CFU']/A_SKIN; out['N_total']=(out['M_G']+out['M_A'])*p['c_CFU']
    return out

def post_removal_rate(p):
    den=max(p['a_w_min']-p['a_w_lethal'],1e-9)
    g=clip01((p['a_w_min']-A_W_SKIN)/den)**2
    return K_D0+p['k_desicc']*g+p['k_starv']+p['k_host']+p['k_shed']+p['k_strip']

def time_to_floor(sigma,p):
    if sigma<=p['floor']: return p['T_gel']/24
    return (p['T_gel']+math.log(sigma/p['floor'])/post_removal_rate(p))/24

def cross(t,y,thr):
    idx=np.where(y<=thr)[0]
    if not len(idx): return np.nan
    i=idx[0]
    if i==0: return float(t[0])
    return float(t[i-1]+(thr-y[i-1])*(t[i]-t[i-1])/(y[i]-y[i-1]))



# ---------- 15-input GSA ----------
GSA_SPECS = OrderedDict([
    ('k_desicc',(0.0015,0.15,'log')),
    ('k_strip',(0.001,0.10,'log')),
    ('k_att',(0.002,0.20,'log')),
    ('k_starv',(0.0005,0.05,'log')),
    ('k_host',(0.0005,0.05,'log')),
    ('k_shed',(0.0004,0.04,'log')),
    ('A_max',(1e-5,1e-3,'log')),
    ('K_cap',(1.2,12.0,'log')),
    ('c_CFU',(1e12/3,3e12,'log')),
    ('T_gel',(24.0,72.0,'uniform')),
    ('k_store',(0.0090,0.0156,'log')),
    ('V_nonwater',(0.11,0.225,'uniform')),
    ('k_E',(0.059,0.236,'log')),
    ('a_w_min',(0.90,0.963,'uniform')),
    ('a_w_lethal',(0.80,0.89,'uniform')),
])

def transform_unit(U,names=None):
    U=np.asarray(U,float)
    if U.ndim==1: U=U[None,:]
    names=list(GSA_SPECS) if names is None else list(names)
    out={}
    for j,n in enumerate(names):
        lo,hi,dist=GSA_SPECS[n]
        if dist=='log': out[n]=np.exp(np.log(lo)+U[:,j]*(np.log(hi)-np.log(lo)))
        else: out[n]=lo+U[:,j]*(hi-lo)
    return out

def batch_rates(V,MG,MA,MS,NL,P):
    V=np.maximum(V,P['V_nonwater']+1e-10); MG=np.maximum(MG,0); MA=np.maximum(MA,0); MS=np.maximum(MS,0); NL=np.maximum(NL,0)
    nw=np.maximum((V-P['V_nonwater'])*RHO_W/MW_H2O,1e-15)
    nosm=N_GLYCEROL+MS/MW_FOS+NL+N_NA_LAC+N_CA+N_GLUC+N_K+N_GLYC+N_NA_ALG
    aw=nw/(nw+nosm)
    VL=np.maximum(V,1e-12)/1000
    strong=(N_NA_LAC+2*N_CA+N_K-2*N_GLYC)/VL
    weak=(NL+N_GLUC)/VL
    H=np.maximum(KA_LAC*(weak/strong-1),1e-12)
    pH=-np.log10(H); LT=NL/VL; HL=LT*H/(KA_LAC+H)
    B=1000*MG/V; S=1000*MS/V
    faw=np.sqrt(np.clip((aw-P['a_w_min'])/(1-P['a_w_min']),0,1))
    nph=(pH-PH_MIN)*(pH-PH_MAX); dph=nph-(pH-PH_OPT)**2
    fph=np.clip(np.divide(nph,dph,out=np.zeros_like(nph),where=dph!=0),0,1)
    fhl=1/(1+HL/K_HL)
    mu=MU_OPT*F_T*S/(K_S+S)*np.maximum(0,1-B/P['K_cap'])*faw*fph*fhl
    den=np.maximum(P['a_w_min']-P['a_w_lethal'],1e-9)
    g=np.clip((P['a_w_min']-aw)/den,0,1)**2
    kd=K_D0+P['k_desicc']*g+K_ACID*HL
    attach=P['k_att']*MG*np.maximum(0,1-MA/P['A_max'])
    dV=J_SKIN*(1-aw)-P['k_E']*np.maximum(0,aw-RH)
    dMG=(mu-kd)*MG-attach
    dMA=attach-kd*MA
    dMS=-(mu/Y_XS+M_S_MAINT)*MG
    dNL=Y_LA_S*(mu*(1/Y_XS-1)+M_S_MAINT)*MG
    return dV,dMG,dMA,dMS,dNL

def batch_endpoints(U,shelf_days=180.0,dt=0.1):
    U=np.asarray(U,float); n=len(U); P=transform_unit(U)
    # fill constants in arrays
    for k,v in BASE.items():
        if k not in P: P[k]=np.full(n,float(v))
    V=np.full(n,V0); MG=M0_PRE_STORAGE*np.exp(-P['k_store']*shelf_days); MA=np.zeros(n); MS=np.full(n,MS0); NL=np.full(n,NL0)
    t=0.0; maxT=float(np.max(P['T_gel']))
    while t < maxT-1e-12:
        h=np.clip(P['T_gel']-t,0,dt)
        active=h>0
        if not np.any(active): break
        k1=batch_rates(V,MG,MA,MS,NL,P)
        Vm=V+0.5*h*k1[0]; MGm=MG+0.5*h*k1[1]; MAm=MA+0.5*h*k1[2]; MSm=MS+0.5*h*k1[3]; NLm=NL+0.5*h*k1[4]
        k2=batch_rates(Vm,MGm,MAm,MSm,NLm,P)
        V += h*k2[0]; MG += h*k2[1]; MA += h*k2[2]; MS += h*k2[3]; NL += h*k2[4]
        t += dt
    sigma=MA*P['c_CFU']/A_SKIN
    den=np.maximum(P['a_w_min']-P['a_w_lethal'],1e-9)
    gskin=np.clip((P['a_w_min']-A_W_SKIN)/den,0,1)**2
    kpost=K_D0+P['k_desicc']*gskin+P['k_starv']+P['k_host']+P['k_shed']+P['k_strip']
    tfloor=P['T_gel']/24 + np.maximum(0,np.log(np.maximum(sigma,1e-300)/BASE['floor']))/(24*kpost)
    # if sigma <= floor, time is Tgel/24, which formula with max handles
    return {'t_floor_days':tfloor,'sigma_removal':sigma,'M_A_removal':MA,'k_post':kpost,'params':P}



In [ ]:
# Cross-model synchronization self-check
assert len(GSA_SPECS) == 15, f"Expected 15 GSA inputs, found {len(GSA_SPECS)}"
assert set(["K_cap", "c_CFU", "a_w_lethal", "A_max", "k_shed", "V_nonwater"]).issubset(GSA_SPECS)
print("SYNC CHECK: normalized Hydrogel cargo interface + 15-input prior set are active.")


## 1. Nominal simulation and reproduction check

In [ ]:
hydro = simulate_hydrogel()
p = hydro["params"]

def crossing_time(t, y, threshold):
    t = np.asarray(t); y = np.asarray(y)
    idx = np.where(y <= threshold)[0]
    if len(idx) == 0:
        return np.nan
    i = idx[0]
    if i == 0:
        return float(t[0])
    return float(t[i-1] + (threshold-y[i-1])*(t[i]-t[i-1])/(y[i]-y[i-1]))

i_peak = int(np.argmax(hydro["N_total"]))
sigma48 = float(hydro["sigma_A"][-1])
nominal_t_floor = time_to_floor(sigma48, p)

summary = {
    "a_w = 0.90 crossing (h)": crossing_time(hydro["t_h"], hydro["aw"], 0.90),
    "a_w = 0.85 crossing (h)": crossing_time(hydro["t_h"], hydro["aw"], 0.85),
    "peak time (h)": hydro["t_h"][i_peak],
    "peak total CFU": hydro["N_total"][i_peak],
    "peak increase (%)": 100*(hydro["N_total"][i_peak]/hydro["N_total"][0]-1),
    "attached density at 48 h (CFU cm^-2)": sigma48,
    "FOS consumed (mg)": 1000*(MS0-hydro["M_S"][-1]),
    "pH at 48 h": hydro["pH"][-1],
    "nominal time-to-floor (d)": nominal_t_floor,
}
pd.Series(summary)

## 2. Core nominal figures

In [ ]:
fig, ax = plt.subplots(figsize=(8,5))
ax.plot(hydro["t_h"], hydro["aw"])
ax.axhline(0.90, ls="--")
ax.axhline(0.85, ls="--")
ax.axhline((J_SKIN + BASE["k_E"]*RH)/(J_SKIN+BASE["k_E"]), ls=":")
ax.set_xlabel("Time after application (h)")
ax.set_ylabel("Water activity")
ax.set_title("Nominal hydrogel water activity")
fig.tight_layout(); plt.show()

fig, ax = plt.subplots(figsize=(8,5))
ax.plot(hydro["t_h"], hydro["N_total"])
ax.set_xlabel("Time after application (h)")
ax.set_ylabel("Total viable population (CFU)")
ax.set_title("Total viable population during the hydrogel phase")
fig.tight_layout(); plt.show()

fig, ax1 = plt.subplots(figsize=(8,5))
ax2 = ax1.twinx()
ax1.plot(hydro["t_h"], hydro["N_G"], label="Gel-held cells")
ax2.plot(hydro["t_h"], hydro["sigma_A"], ls="--", label="Attached density")
ax1.set_xlabel("Time after application (h)")
ax1.set_ylabel("Gel-held population (CFU)")
ax2.set_ylabel("Attached density (CFU cm$^{-2}$)")
ax1.set_title("Fresh two-compartment trajectory")
fig.tight_layout(); plt.show()

## 3. Post-removal persistence and seven-day feasibility envelope

In [ ]:
t_days = np.linspace(BASE["T_gel"]/24, 11, 300)
kpost_nom = post_removal_rate(BASE)
sigma_tail = sigma48*np.exp(-kpost_nom*(24*t_days-BASE["T_gel"]))

fig, ax = plt.subplots(figsize=(8,5))
ax.semilogy(t_days, sigma_tail)
ax.axhline(BASE["floor"], ls="--")
ax.axvline(nominal_t_floor, ls=":")
ax.set_xlabel("Time since application (d)")
ax.set_ylabel("Attached density (CFU cm$^{-2}$)")
ax.set_title("Nominal post-removal persistence")
fig.tight_layout(); plt.show()

T_target_h = 7*24
sigma_required_7d = BASE["floor"] * np.exp(kpost_nom*(T_target_h-BASE["T_gel"]))
print(f"Seven-day removal-density requirement: {sigma_required_7d:.3e} CFU cm^-2")
print(f"Nominal k_post: {kpost_nom:.4f} h^-1")

## 4. Carrying-capacity structural sweep

In [ ]:
kcap_rows=[]
fig, ax = plt.subplots(figsize=(8,5))
for kc in [1.2, 3.0, 12.0]:
    h = simulate_hydrogel({"K_cap":kc})
    ratio = h["N_total"]/h["N_total"][0]
    i = int(np.argmax(ratio))
    tf = time_to_floor(h["sigma_A"][-1], h["params"])
    kcap_rows.append({
        "K_cap (g/L)":kc,
        "peak growth (%)":100*(ratio[i]-1),
        "peak time (h)":h["t_h"][i],
        "FOS consumed (mg)":1000*(MS0-h["M_S"][-1]),
        "pH at 48 h":h["pH"][-1],
        "sigma_A at 48 h":h["sigma_A"][-1],
        "time-to-floor (d)":tf,
    })
    ax.plot(h["t_h"], ratio, label=f"K_cap = {kc:g} g L$^{{-1}}$")
ax.set_xlabel("Time after application (h)")
ax.set_ylabel("Total viable population / starting population")
ax.set_title("Carrying-capacity structural sensitivity")
ax.legend(); fig.tight_layout(); plt.show()
pd.DataFrame(kcap_rows)

## 5. Storage-conditioned reruns

In [ ]:
storage_cases = [
    ("Fresh", 0.0, 0.0),
    ("180 d protected", 0.0090, 180.0),
    ("180 d baseline", 0.0156, 180.0),
]
rows=[]
fig, ax = plt.subplots(figsize=(8,5))
for label, ks, shelf in storage_cases:
    h = simulate_hydrogel({"k_store":ks}, shelf_days=shelf)
    p = h["params"]
    sigma0 = h["sigma_A"][-1]
    tf = time_to_floor(sigma0,p)
    rows.append({"Scenario":label,"sigma_A at 48 h":sigma0,"time-to-floor (d)":tf})
    td = np.linspace(p["T_gel"]/24,10,300)
    sig = sigma0*np.exp(-post_removal_rate(p)*(24*td-p["T_gel"]))
    ax.semilogy(td,sig,label=label)
ax.axhline(BASE["floor"],ls="--")
ax.set_xlabel("Time since application (d)")
ax.set_ylabel("Attached density (CFU cm$^{-2}$)")
ax.set_title("Nominal storage-conditioned persistence")
ax.legend(); fig.tight_layout(); plt.show()
pd.DataFrame(rows)

## 6. Endpoint-definition sensitivity

In [ ]:
floors = np.array([1e3,1e4,1e5,1e6])
times = [time_to_floor(sigma48, BASE | {"floor":float(f)}) for f in floors]
fig, ax = plt.subplots(figsize=(8,5))
ax.barh([r"$10^3$",r"$10^4$",r"$10^5$",r"$10^6$"],times)
ax.set_xlabel("Time above floor from application (d)")
ax.set_ylabel("Functional floor (CFU cm$^{-2}$)")
ax.set_title("Endpoint-definition sensitivity")
fig.tight_layout(); plt.show()
pd.DataFrame({"floor":floors,"time_days":times})

## 7. Biomass-to-CFU structural sensitivity

In [ ]:
c_values = [1e12/3, 1e12, 3e12]
rows=[]
# reporting-only: use the nominal biomass trajectory, only change conversion
M_A48_nom = float(hydro["M_A"][-1])
for c in c_values:
    p=BASE.copy(); p["c_CFU"]=c
    sigma=M_A48_nom*c/A_SKIN
    rows.append({"mode":"fixed biomass trajectory","c_CFU":c,"sigma_A":sigma,"time_days":time_to_floor(sigma,p)})
# inoculum-consistent: keep N0 = 1e9 CFU fixed, so initial biomass changes
for c in c_values:
    p=BASE.copy(); p["c_CFU"]=c
    h=simulate_hydrogel(p,initial_biomass_g=1e9/c)
    rows.append({"mode":"fixed 1e9 CFU inoculum","c_CFU":c,"sigma_A":h["sigma_A"][-1],"time_days":time_to_floor(h["sigma_A"][-1],p)})
cfu_df=pd.DataFrame(rows)
fig, ax = plt.subplots(figsize=(8,5))
for mode,g in cfu_df.groupby("mode"):
    ax.plot(g["c_CFU"],g["time_days"],marker="o",label=mode)
ax.set_xscale("log")
ax.set_xlabel(r"$c_{CFU}$ (CFU g$^{-1}$)")
ax.set_ylabel("Time-to-floor (d)")
ax.set_title("Biomass-to-CFU structural sensitivity")
ax.legend(); fig.tight_layout(); plt.show()
cfu_df

## 8. Equilibrium water-activity map

In [ ]:
tewl_grid=np.linspace(5,70,100)
rh_grid=np.linspace(0.20,0.85,100)
TT,RR=np.meshgrid(tewl_grid,rh_grid)
J=TT*(A_SKIN*1e-4)
AW=(J+BASE["k_E"]*RR)/(J+BASE["k_E"])
fig,ax=plt.subplots(figsize=(8,5))
cf=ax.contourf(TT,100*RR,AW,levels=14)
ax.contour(TT,100*RR,AW,levels=[0.85,0.90])
ax.scatter([TEWL],[100*RH])
ax.set_xlabel(r"TEWL (g m$^{-2}$ h$^{-1}$)")
ax.set_ylabel("Relative humidity (%)")
ax.set_title("Equilibrium water activity under Eq. 1")
fig.colorbar(cf,ax=ax,label="Equilibrium water activity")
fig.tight_layout(); plt.show()

## 9. 15-input Morris screening

In [ ]:
def morris_screen(r=80, levels=8, seed=20260928, shelf_days=180.0):
    names=list(GSA_SPECS)
    d=len(names)
    rng=np.random.default_rng(seed)
    delta=levels/(2*(levels-1))
    base_grid=np.arange(0,levels//2)/(levels-1)
    effects={n:[] for n in names}
    for _ in range(r):
        x0=rng.choice(base_grid,size=d)
        order=rng.permutation(d)
        points=[x0.copy()]
        x=x0.copy()
        for j in order:
            x=x.copy(); x[j]+=delta; points.append(x)
        points=np.asarray(points)
        flips=rng.integers(0,2,size=d).astype(bool)
        points[:,flips]=1-points[:,flips]
        y=batch_endpoints(points,shelf_days)["t_floor_days"]
        for step,j in enumerate(order):
            dx=points[step+1,j]-points[step,j]
            effects[names[j]].append((y[step+1]-y[step])/dx)
    rows=[]
    for n in names:
        e=np.asarray(effects[n])
        rows.append({"Parameter":n,"mu_star":np.mean(np.abs(e)),"sigma":np.std(e,ddof=1)})
    return pd.DataFrame(rows).sort_values("mu_star",ascending=False)

morris=morris_screen()
fig,ax=plt.subplots(figsize=(8,6))
p=morris.sort_values("mu_star")
ax.barh(p["Parameter"],p["mu_star"])
ax.set_xlabel(r"Morris $\mu^*$ (days per normalized range)")
ax.set_title("Morris screening")
fig.tight_layout(); plt.show()
morris

## 10. 15-input Sobol/Jansen analysis

In [ ]:
def sobol_jansen(N=4096, seed=20260928, shelf_days=180.0, output="t_floor_days"):
    names=list(GSA_SPECS)
    d=len(names)
    m=int(round(math.log2(N)))
    if 2**m != N:
        raise ValueError("N must be a power of two for random_base2().")
    U=qmc.Sobol(d=2*d,scramble=True,seed=seed).random_base2(m)
    A=U[:,:d]; B=U[:,d:]
    YA=batch_endpoints(A,shelf_days)[output]
    YB=batch_endpoints(B,shelf_days)[output]
    var=np.var(np.concatenate([YA,YB]),ddof=1)
    S1=np.empty(d); ST=np.empty(d)
    for i in range(d):
        AB=A.copy(); AB[:,i]=B[:,i]
        YAB=batch_endpoints(AB,shelf_days)[output]
        S1[i]=np.mean(YB*(YAB-YA))/var
        ST[i]=0.5*np.mean((YA-YAB)**2)/var
    return pd.DataFrame({"Parameter":names,"S1":S1,"ST":ST}).sort_values("ST",ascending=False)

sobol_persistence=sobol_jansen(N=4096,output="t_floor_days")
plot_df=sobol_persistence.sort_values("ST")
y=np.arange(len(plot_df)); h=0.38
fig,ax=plt.subplots(figsize=(8,6))
ax.barh(y-h/2,plot_df["S1"],height=h,label="First-order")
ax.barh(y+h/2,plot_df["ST"],height=h,label="Total-order")
ax.set_yticks(y); ax.set_yticklabels(plot_df["Parameter"])
ax.set_xlabel("Sobol sensitivity index")
ax.set_title("Persistence sensitivity: 15-input GSA")
ax.legend(); fig.tight_layout(); plt.show()
sobol_persistence

## 11. Sobol sensitivity of physical attached biomass at removal

In [ ]:
sobol_attachment=sobol_jansen(N=4096,output="M_A_removal")
p=sobol_attachment.sort_values("ST")
fig,ax=plt.subplots(figsize=(8,6))
ax.barh(p["Parameter"],p["ST"])
ax.set_xlabel("Total-order Sobol index")
ax.set_title("Sensitivity of attached biomass at removal")
fig.tight_layout(); plt.show()
sobol_attachment

## 12. Matched fresh- and storage-conditioned uncertainty propagation

In [ ]:
N_GLOBAL=4096
U=qmc.Sobol(d=len(GSA_SPECS),scramble=True,seed=20260928).random_base2(12)
fresh=batch_endpoints(U,shelf_days=0.0)
stored=batch_endpoints(U,shelf_days=180.0)

def summarize_uncertainty(label,out):
    t=out["t_floor_days"]
    return {
        "Scenario":label,
        "median_days":np.median(t),
        "q2.5_days":np.quantile(t,0.025),
        "q97.5_days":np.quantile(t,0.975),
        "fraction_ge_7":np.mean(t>=7),
        "median_sigma_A":np.median(out["sigma_removal"]),
    }

uncertainty_summary=pd.DataFrame([
    summarize_uncertainty("Fresh-conditioned",fresh),
    summarize_uncertainty("180 d storage-conditioned",stored),
])

fig,ax=plt.subplots(figsize=(8,5))
bins=np.linspace(0,max(np.quantile(fresh["t_floor_days"],0.995),np.quantile(stored["t_floor_days"],0.995)),35)
ax.hist(fresh["t_floor_days"],bins=bins,histtype="step",linewidth=2,label="Fresh-conditioned")
ax.hist(stored["t_floor_days"],bins=bins,histtype="step",linewidth=2,label="180 d storage-conditioned")
ax.axvline(7,ls="--",label="7 d target")
ax.axvline(np.median(fresh["t_floor_days"]),ls=":",label=f"Fresh median {np.median(fresh['t_floor_days']):.2f} d")
ax.axvline(np.median(stored["t_floor_days"]),ls="-.",label=f"Storage median {np.median(stored['t_floor_days']):.2f} d")
ax.set_xlabel(r"Time above $10^4$ CFU cm$^{-2}$ floor (d)")
ax.set_ylabel("Screening samples")
ax.set_title("Fresh vs storage-conditioned global uncertainty")
ax.legend(); fig.tight_layout(); plt.show()

print(uncertainty_summary.to_string(index=False))
print(f"Median storage penalty (paired): {np.median(fresh['t_floor_days']-stored['t_floor_days']):.2f} d")
print(f"Median k_post: {np.median(stored['k_post']):.4f} h^-1")
print(f"Fraction of k_post draws above nominal 0.0398 h^-1: {np.mean(stored['k_post']>0.0398):.3f}")

## 13. Optional convergence check for Sobol total-order indices

In [ ]:
convergence=[]
for N in [1024,2048,4096]:
    s=sobol_jansen(N=N,output="t_floor_days")
    for _,row in s.iterrows():
        convergence.append({"N":N,"Parameter":row["Parameter"],"ST":row["ST"]})
conv_df=pd.DataFrame(convergence)
conv_df.pivot(index="Parameter",columns="N",values="ST").sort_values(4096,ascending=False)

## 14. Relative cargo interface (requires the actual upstream promoter trajectory)

In [ ]:
def simulate_relative_cargo(t_flare_h, ptrxb_values, hydro=None, k_deg=K_DEG, k_ves_rel=1.0, y_lys_rel=1.0):
    """Relative Hydrogel cargo interface; no absolute EV dose is implied."""
    t_flare_h=np.asarray(t_flare_h,float)
    ptrxb_values=np.asarray(ptrxb_values,float)
    if len(t_flare_h)!=len(ptrxb_values):
        raise ValueError("t_flare_h and ptrxb_values must have the same length")
    if np.any(np.diff(t_flare_h)<=0):
        raise ValueError("t_flare_h must be strictly increasing")
    if hydro is None:
        hydro=simulate_hydrogel()
    t=hydro["t_h"]
    P=np.interp(t,t_flare_h,ptrxb_values)
    fsec=hydro["f_aw_sec"]

    def rhs(ti,r):
        return k_deg*(np.interp(ti,t,P)*np.interp(ti,t,fsec)-r[0])
    rsol=solve_ivp(rhs,[t[0],t[-1]],[0.0],t_eval=t,method="LSODA",rtol=1e-8,atol=1e-10)
    r=rsol.y[0]
    theta=np.clip(r,0,1)
    phi_sec=k_ves_rel*(hydro["mu"]*(1/Y_XS-1)+M_S_MAINT)*hydro["f_aw_sec"]*(hydro["M_G"]+hydro["M_A"])
    phi_lys=y_lys_rel*hydro["k_d_G"]*(hydro["M_G"]+hydro["M_A"])
    # conservative useful route = active secretion only
    loaded_flux=theta*phi_sec
    unloaded_flux=(1-theta)*phi_sec
    dt=np.diff(t)
    Q_loaded=np.zeros_like(t); Q_unloaded=np.zeros_like(t)
    Q_loaded[1:]=np.cumsum(0.5*(loaded_flux[1:]+loaded_flux[:-1])*dt)
    Q_unloaded[1:]=np.cumsum(0.5*(unloaded_flux[1:]+unloaded_flux[:-1])*dt)
    return pd.DataFrame({
        "time_h":t,"P_trxB":P,"r":r,"theta":theta,
        "phi_sec_rel":phi_sec,"phi_lys_rel":phi_lys,
        "Q_loaded_rel":Q_loaded,"Q_unloaded_rel":Q_unloaded,
    })

# Example after exporting the real upstream trajectory:
# flare = pd.read_csv('/content/P_trxB.csv')
# cargo = simulate_relative_cargo(flare['time_h'], flare['P_trxB'])
# cargo.head()

## 15. Export result tables (optional)

In [ ]:
# Uncomment if you want CSV files from Colab.
# pd.DataFrame(kcap_rows).to_csv('/content/kcap_sweep.csv', index=False)
# cfu_df.to_csv('/content/cfu_conversion_sweep.csv', index=False)
# morris.to_csv('/content/morris_screening.csv', index=False)
# sobol_persistence.to_csv('/content/sobol_persistence.csv', index=False)
# sobol_attachment.to_csv('/content/sobol_attachment.csv', index=False)
# uncertainty_summary.to_csv('/content/uncertainty_summary.csv', index=False)
print('Notebook ready. Run cells from top to bottom in Google Colab.')